## 1. Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import TimeSeriesSplit

## 2. Load Data

In [ ]:
# Set a variable to the collected data
DATA_PATH = '/workspace/data/metrics.csv'

# Pulling in our data, setting the index to the timestamp, and sorting by the index
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
df = df.set_index('timestamp').sort_index()

# Memory utilization %
df['mem_util_percent'] = (df['mem_used_gb'] / (df['mem_used_gb'] + df['mem_available_gb'])) * 100

# Drop any rows with empty values
df = df.dropna()

## 3. Generate Anomaly Labels

In [ ]:
# Define the columns we want to focus on
features = [
    'cpu_percent',
    'mem_util_percent',
    'gpu_util_percent',
    'gpu_mem_used_gb'
]

# Create a new dataframe with just the columns we want
X = df[features].copy()

# Create out IsolationForest
iso_forest = IsolationForest(
    contamination=0.03, # Standard value here, worked well in our previous experiments with this data
    random_state=42, # as per convention
    n_estimators=100 # default
)

# Generate our list of anomalies and add them to the dataframe
df['anomaly_score'] = iso_forest.fit_predict(X)
df['anomaly'] = df['anomaly_score'] == -1


## 4. Feature Engineering

In [ ]:
# Create new columns from the rolling mean of our select columns.
# Window=10 means each window looks at the last ten rows of data, ten minutes of time for our dataset
df['cpu_rolling_mean'] = df['cpu_percent'].rolling(window=10).mean()
df['mem_rolling_mean'] = df['mem_util_percent'].rolling(window=10).mean()
df['gpu_rolling_mean'] = df['gpu_util_percent'].rolling(window=10).mean()
df['gpu_mem_rolling_mean'] = df['gpu_mem_used_gb'].rolling(window=10).mean()

# Drop any rows with empty values
df = df.dropna()

# Define a list of our rolling means
rolling_features = [
    'cpu_rolling_mean',
    'mem_rolling_mean',
    'gpu_rolling_mean',
    'gpu_mem_rolling_mean'
]

# Create a new dataframe containing our rolling mean columns
X = df[rolling_features].copy()

# Set y to the boolean series of the anomaly column
y = df['anomaly']

## 5. Building the pipeline

In [ ]:
# Build a pipeline to fit to the training data, transform the data to be on the same scale (some data might be a percentage while others are raw values)
# and then train the model on the scaled data
pipe = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced'))

# Define our time splits.  5 splits is the default here.  We use time splits instead of a test train split to fit our time series data
# This prevents the model from "learning from the future" and only uses past data to train
tscv = TimeSeriesSplit(n_splits=5)

# Make a simple variable to hold our results
results = []

# Main loop.  This breaks our data into our Time Splits and then runs each split through the pipeline before predicting off that data
# Prediction accuracy measurements are then added to our results variable 
for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)
    results.append({
        'precision': precision_score(y_test, predictions),
        'recall': recall_score(y_test, predictions),
        'f1': f1_score(y_test, predictions)
        })

# Convert our results variable into a DataFrame
results_df = pd.DataFrame(results)


/usr/local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## 6. Summarize the results

In [ ]:
# This prints our the mean and standard deviation for the results, giving us the evaluation metrics for model performance
print(results_df.mean())
print(results_df.std())

precision    0.287071
recall       0.281980
f1           0.168925
dtype: float64
precision    0.410526
recall       0.349121
f1           0.153827
dtype: float64


## 7. Confusion Matrix

In [ ]:
# This prints out the confusion matrix from the last time split
# Matrix interpretation is as follows:
# [[True Negative, False Positives]
#  [False Negatives, True Positives]]
print(confusion_matrix(y_test, predictions))

[[1838   51]
 [  12    5]]


## Observations
This notebook took our collected homelab data, prepped the data for using in a Logistic Regression pipeline, and trained the model to detect anomalies.  Expected output will be poor due to training a supervised model with unsupervised data.
- Results showed the model performed poorly with the data set we had, originally finding all data was "normal" and not anomolous, as expected.
- This was addressed by adding class_weight=balanced to the model to have it weigh missed anomalies more in the data to allow us to find them.  The final result was mixed.  A lower precision score (82 false positives) and a middling recall score (3 of 14 anomalies found) of the final run are an example of this.  Over all, the model only successfully found 44% of the true positives
- I believe this was primarily caused by the model needing supervised data and our anomaly label was generated by an unsupervised IsolationForest.  Models that require supervised data may underperform when given unsupervised data due to mislabeling and the lack of clear, defined outputs that supervised learning relies on during training
- Next steps should be to generate some supervised data for the model to compare how it affects the model's performance.  Additionally, it would be good to try a Random Forest as well